# Research 04 — 注意力机制手写（SDPA、多头、KV cache）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Papa-Panda/field-notes/blob/main/pytorch/research/04-attention.ipynb)

- **日期**：2026-10-02
- **定位**：Research PyTorch 轮次的模型结构面。注意力是唯一一个面试官敢让你从零写的组件 —— 能手写、能和官方实现数值对齐，才算过关。
- **环境**：纯 PyTorch，CPU 即可。

配套笔记：[04-attention.md](04-attention.md)


## 今日目标

1. 手写 scaled dot-product attention 并与 `F.scaled_dot_product_attention` 数值对齐。
2. 手写 causal mask、多头注意力，并与 `nn.MultiheadAttention` 做权重等价验证。
3. 实现 KV cache 的增量解码，验证与全序列前向逐步等价。

公式先对齐：$ \mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V $ 。除 $ \sqrt{d_k} $ 是因为点积方差随 $ d_k $ 线性增长，不缩放 softmax 会饱和、梯度消失。


## 1. 手写 SDPA vs 官方实现

In [ ]:
import torch
import torch.nn.functional as F

def manual_sdpa(Q, K, V, mask=None):
    """Q,K,V: (..., T, d)。mask: 可广播到 (..., T, T)，加在 softmax 之前。"""
    d_k = Q.size(-1)
    scores = Q @ K.transpose(-2, -1) / (d_k ** 0.5)   # ← 缩放是 √d_k，不是 d_k
    if mask is not None:
        scores = scores + mask                        # ← mask 在 softmax 之前
    weights = torch.softmax(scores, dim=-1)
    return weights @ V

torch.manual_seed(0)
Q = torch.randn(2, 6, 8); K = torch.randn(2, 6, 8); V = torch.randn(2, 6, 8)
mine = manual_sdpa(Q, K, V)
ref = F.scaled_dot_product_attention(Q, K, V)
print('SDPA 与官方实现最大差:', (mine - ref).abs().max().item(), '（<1e-6 即对齐）')


## 2. Causal mask：位置 $ t $ 只能看 $ \le t $

In [ ]:
T = 6
mask = torch.full((T, T), float('-inf')).triu(diagonal=1)  # 上三角 -inf
out_mine = manual_sdpa(Q, K, V, mask=mask)
out_ref = F.scaled_dot_product_attention(Q, K, V, is_causal=True)
print('causal 版与官方 is_causal 最大差:', (out_mine - out_ref).abs().max().item())

# 泄漏自检：改最后一个 token 的输入，前 5 个位置的输出必须不变
Q2, K2, V2 = Q.clone(), K.clone(), V.clone()
K2[:, -1] += 100.0; V2[:, -1] += 100.0
out2 = manual_sdpa(Q2, K2, V2, mask=mask)
print('改最后一个 token 后，前 5 个位置输出最大变化:', (out_mine[:, :-1] - out2[:, :-1]).abs().max().item(),
      '（≈0 = 无未来信息泄漏）')


## 3. 多头注意力：从零写，与官方模块等价

$ d_{model} $ 切成 $ h $ 份，每份 $ d_k = d_{model}/h $ ，各做一遍 SDPA 再拼回。维度顺序 (batch, heads, seq, head_dim) 不能错 —— 这是能跑但结果全错的头号坑。


In [ ]:
class MultiHeadAttention(torch.nn.Module):
    def __init__(self, d_model=32, num_heads=4):
        super().__init__()
        assert d_model % num_heads == 0
        self.h, self.d_k = num_heads, d_model // num_heads
        self.qkv = torch.nn.Linear(d_model, 3 * d_model)   # 一个大投影：工程实现如此
        self.out = torch.nn.Linear(d_model, d_model)

    def forward(self, x, causal=False):
        B, T, D = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        # (B, T, D) → (B, h, T, d_k)
        def split(t):
            return t.view(B, T, self.h, self.d_k).transpose(1, 2)
        q, k, v = split(q), split(k), split(v)
        mask = torch.full((T, T), float('-inf'), device=x.device).triu(1) if causal else None
        y = manual_sdpa(q, k, v, mask=mask)
        y = y.transpose(1, 2).contiguous().view(B, T, D)   # 拼回 (B, T, D)
        return self.out(y)

torch.manual_seed(0)
mine = MultiHeadAttention(32, 4)
refm = torch.nn.MultiheadAttention(32, 4, batch_first=True, dropout=0.0)
# 把我的权重灌进官方模块：qkv 行块 = [Wq; Wk; Wv]
with torch.no_grad():
    refm.in_proj_weight.copy_(mine.qkv.weight)
    refm.in_proj_bias.copy_(mine.qkv.bias)
    refm.out_proj.weight.copy_(mine.out.weight)
    refm.out_proj.bias.copy_(mine.out.bias)
mine.eval(); refm.eval()
x = torch.randn(2, 5, 32)
with torch.no_grad():
    a = mine(x)
    b, _ = refm(x, x, x)
print('手写 MHA 与 nn.MultiheadAttention 最大差:', (a - b).abs().max().item(), '（<1e-5 即等价）')


## 4. KV cache：增量解码与全序列前向等价

推理时每步只喂一个新 token，K/V 全部缓存复用。验证：逐步解码每一步的输出 == 全前向对应位置的输出。


In [ ]:
# 简化验证：用 SDPA 层面演示 KV cache 等价性 ——
# 第 t 步只算新 token 的 q，k/v 取缓存的前 t+1 个
torch.manual_seed(0)
T, D = 8, 16
x = torch.randn(1, T, D)
Wq, Wk, Wv = [torch.nn.Linear(D, D, bias=False) for _ in range(3)]
Q, K, V = Wq(x), Wk(x), Wv(x)
mask = torch.full((T, T), float('-inf')).triu(1)
full = manual_sdpa(Q, K, V, mask=mask)

k_cache, v_cache = [], []
step_outs = []
for t in range(T):
    k_cache.append(K[:, t:t+1]); v_cache.append(V[:, t:t+1])
    Kcat = torch.cat(k_cache, dim=1); Vcat = torch.cat(v_cache, dim=1)
    step_outs.append(manual_sdpa(Q[:, t:t+1], Kcat, Vcat))   # 无 mask：缓存里只有 ≤t 的位置
cached = torch.cat(step_outs, dim=1)
print('逐步解码（KV cache）与全前向最大差:', (full - cached).abs().max().item(), '（≈0 即等价）')
print('每步 attention 规模: 第 t 步只需算 1 个 query × (t+1) 个 key，全前向是 T×T —— 这就是 KV cache 省的计算')


## 5. Debug 检查清单（面试给代码找 bug 时的对号入座）

- **症状：loss 低得不真实** —— mask 乘在 softmax 之后还不 renormalize，未来信息泄漏。
  **修复**：mask 在 softmax **之前**加（禁止位置设 $-\infty$ 再 softmax）；用上面的"改最后一个 token"自检验证无泄漏。
- **症状：shape 不报错、结果全错** —— (B, T, h, d_k) 与 (B, h, T, d_k) 转置错。
  **修复**：和官方实现对拍是唯一抓法（第 2、3 节的对照代码），别指望 shape 检查能救。
- **症状：注意力分布系统性偏平或偏尖** —— 缩放分母用了 $ \sqrt{d_{model}} $ 。
  **修复**：分母是每个 head 的 $ d_k $ ；head 数越多，用 $ d_{model} $ 偏得越远。
- **症状：全被 mask 的行（如 padding 行）漏出均匀权重而不是 NaN/0，症状隐蔽** —— causal mask 用大负数 $-10^9$ 代替 $-\infty$ 。
  **修复**：mask 值用 $-\infty$ （或该 dtype 的最小值）；全 mask 的行要么显式置零，要么保证不参与 loss。

## 小结

- SDPA 三件套：投影、$ 1/\sqrt{d_k} $ 缩放、softmax 前的 mask。
- 多头的命门是维度顺序，对拍官方实现是手写正确性的唯一标准。
- KV cache 与全前向严格等价，每步复杂度从 $ O(T^2) $ 降到 $ O(T) $ ，代价是显存随 $ T $ 线性涨。

**Research 系列完**（01 训练循环 → 02 DPO → 03 PPO/GRPO → 04 注意力）。下一条线：并行基础（TP 手写 linear、ZeRO/FSDP），Day 01 的 DDP 在那条线等着。
